In [7]:
import pandas as pd
import numpy as np
import re

In [8]:
# Load dataset
df = pd.read_csv("../Raw-Data/CEAS_08.csv")
df.head()

,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


In [9]:
# Displays current columns and shape
print(df.columns)
print(df.shape)
df.info()

Index(['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'], dtype='object')
(39154, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39154 entries, 0 to 39153
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sender    39154 non-null  object
 1   receiver  38692 non-null  object
 2   date      39154 non-null  object
 3   subject   39126 non-null  object
 4   body      39154 non-null  object
 5   label     39154 non-null  int64 
 6   urls      39154 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 2.1+ MB


In [10]:
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].head(3))


--- sender ---
0                     Young Esposito <Young@iworld.de>
1                         Mok <ipline's1983@icable.ph>
2    Daily Top 10 <Karmandeep-opengevl@universalnet...
Name: sender, dtype: object

--- receiver ---
0      user4@gvc.ceas-challenge.cc
1    user2.2@gvc.ceas-challenge.cc
2    user2.9@gvc.ceas-challenge.cc
Name: receiver, dtype: object

--- date ---
0    Tue, 05 Aug 2008 16:31:02 -0700
1    Tue, 05 Aug 2008 18:31:03 -0500
2    Tue, 05 Aug 2008 20:28:00 -1200
Name: date, dtype: object

--- subject ---
0    Never agree to be a loser
1       Befriend Jenna Jameson
2         CNN.com Daily Top 10
Name: subject, dtype: object

--- body ---
0    Buck up, your troubles caused by small dimensi...
1    \nUpgrade your sex and pleasures with these te...
2    >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...
Name: body, dtype: object

--- label ---
0    1
1    1
2    1
Name: label, dtype: int64

--- urls ---
0    1
1    1
2    1
Name: urls, dtype: int64


In [14]:
# Text cleaning function
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x 
# Clean subject and body
df["subject_clean"] = df["subject"].apply(clean_text) if "subject" in df.columns else ""
df["body_clean"] = df["body"].apply(clean_text)

# Combine into a single text column
df["email_text"] = (df["subject_clean"].astype(str) + " " + df["body_clean"].astype(str)).str.strip()

# Ensure label is numeric
df["lable"] = df["label"].astype(int)

# Keep only necessary columns
clean_df = df[["email_text", "label"]].copy()

# Removes empty and duplicate entries
clean_df = clean_df.dropna()
clean_df = clean_df[clean_df["email_text"].str.len() > 0]
clean_df = clean_df.drop_duplicates(subset=["email_text"]).reset_index(drop=True)

#verifies dataset
print("cleaned Dataset Shape:", clean_df.shape)
print("\nLabel Distribution:")
print(clean_df["label"].value_counts())

clean_df.head()

cleaned Dataset Shape: (38963, 2)

Label Distribution:
label
1    21674
0    17289
Name: count, dtype: int64


,email_text,label
0,"Never agree to be a loser Buck up, your troubl...",1
1,Befriend Jenna Jameson Upgrade your sex and pl...,1
2,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,1
3,Re: svn commit: r619753 - in /spamassassin/tru...,0
4,SpecialPricesPharmMoreinfo WelcomeFastShipping...,1


In [19]:
clean_df["email_text"].str.contains(
    r"\b(spam|ham|phishing|legitimate)\b",
    case=False,
    na=False
).mean()

C:\Users\Laqua\AppData\Local\Temp\ipykernel_3308\1051205503.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  clean_df["email_text"].str.contains(


np.float64(0.036342170777404204)

In [20]:
clean_df.sample(5)

,email_text,label
34795,Re: [ILUG] iSCSI & Shared Storage Question On ...,0
1627,Order Rolex Replica //atches 0nline! Therefore...,1
34123,[UAI] Call for demo proposals and workshop pap...,0
38636,Science Table of Contents Text for 7 March 200...,0
16502,Superhero needs herbal help too NH Plenty of m...,1


In [21]:
clean_df.to_csv(
    "../Processed-Data/ceas_cleaned.csv",
    index=False
)

print("CEAS dataset cleaned and saved successfully!")

CEAS dataset cleaned and saved successfully!
